In [4]:
import pytao
import os

In [11]:
for model in ("cu_hxr", "cu_sxr"):
    tao = pytao.Tao()
    lcls_lattice_path = os.environ.get("LCLS_LATTICE", '../../../')
    init_path = os.path.join(lcls_lattice_path, "bmad", "models", model, "tao.init")
    tao.init(f"-noplot -init {init_path}")
    
    #Get a list of all the undulator phase shifters
    ps_list = tao.cmd("show lat -no_label_lines -attribute B_MAX PS*")
    ps_list = [row for row in ps_list if "#" not in row] # Filter out all the 'split' elements.
    
    # Create a template that we'll use for each phase shifter overlay.
    overlay_template = "O_{element_name}: overlay = {{\n\t{element_name}[B_MAX]:2*pi/{element_name}[L_PERIOD]*sqrt(2*phase_integral*1E-9/{element_name}[L])}}, var = {{phase_integral}}\n\n"
    
    filename = f"und_phase_shifter_overlays_{model[-3:]}.bmad"
    with open(os.path.join(lcls_lattice_path, "bmad", "overlays", "cu", filename), "w") as f:
        for ps_row in ps_list:
            _, ps_name, _, _, _, _ = ps_row.split()
            f.write(overlay_template.format(element_name=ps_name))